# AWS SQS Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/aws_sqs/sqs_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/aws_sqs/sqs_demo.ipynb)

## Business Scenario

E-commerce and microservice systems publish events to SQS. You need a reliable way to ingest messages and validate them before storage.

## Value Proposition

- Reliable ingestion with validation at the edge
- Contract-based schema enforcement for events
- Quarantine invalid messages instead of losing them

---

## Goals

1. Connect to an SQS queue
2. Stream and validate events
3. Materialize clean records


## 🚀 Step 1: Setup AWS SQS

Before running this notebook, you need:
1. An AWS SQS queue (Standard or FIFO)
2. AWS credentials configured (via `~/.aws/credentials` or IAM role)
3. The queue URL

Set your environment variable:
```bash
export AWS_SQS_QUEUE_URL="https://sqs.us-east-1.amazonaws.com/123456789012/my-queue"
```

## 📝 Step 2: Review the Contract

Our contract defines the expected message schema and quality rules.

In [ ]:
with open('sqs_contract.yaml', 'r') as f:
    print("📄 SQS Contract:")
    print("----------------")
    print(f.read())

## ▶️ Step 3: Start the SQS Consumer

This will connect to your SQS queue and start processing messages.

In [ ]:
from lakelogic.core.streaming_processor import StreamingDataProcessor
import threading
import time

# Initialize processor
processor = StreamingDataProcessor(
    contract="sqs_contract.yaml",
    framework="bytewax"
)

# Start in background
thread = threading.Thread(target=processor.start)
thread.daemon = True
thread.start()

print("🚀 SQS consumer started!")
print("Waiting for order events...")
time.sleep(5)

## 🧪 Step 4: Send Test Messages

Let's send some test order events to the queue.

In [ ]:
import boto3
import json
import os
from datetime import datetime

queue_url = os.getenv("AWS_SQS_QUEUE_URL")
if queue_url:
    sqs = boto3.client('sqs', region_name='us-east-1')
    
    test_orders = [
        {
            "orderId": "ORD-001",
            "customerId": "CUST-123",
            "amount": 99.99,
            "currency": "USD",
            "timestamp": datetime.now().isoformat(),
            "status": "pending"
        },
        {
            "orderId": "ORD-002",
            "customerId": "CUST-456",
            "amount": 149.50,
            "currency": "EUR",
            "timestamp": datetime.now().isoformat(),
            "status": "completed"
        }
    ]
    
    for order in test_orders:
        response = sqs.send_message(
            QueueUrl=queue_url,
            MessageBody=json.dumps(order)
        )
        print(f"📤 Sent order {order['orderId']} - MessageId: {response['MessageId']}")
    
    print("\n✅ Test messages sent! Check LakeLogic logs...")
else:
    print("ℹ️  Set AWS_SQS_QUEUE_URL to send test messages")

## 📊 Step 5: Verify Results

Check the materialized Delta table.

In [ ]:
import polars as pl
import time

# Wait for processing
time.sleep(3)

try:
    df = pl.read_delta("./data/bronze/aws_orders/")
    print("📂 Processed Orders:")
    print(df)
except Exception as e:
    print(f"ℹ️  No data yet: {e}")

## 🎉 Summary

You just:
- ✅ Connected to AWS SQS
- ✅ Validated order events against a contract
- ✅ Materialized events to Delta Lake

This pattern enables **reliable, scalable ingestion** from microservices architectures!